# Privacy and Governance Analysis

Analysis Overview

This project simulates a comprehensive Regulatory & Technical Audit of “NovaCred,” an AI-driven credit scoring system.

In the financial sector, automated decision-making processes significantly impact individuals’ fundamental rights. Therefore, this audit ensures that NovaCred’s data pipeline and algorithmic outcomes align with the highest European standards: the General Data Protection Regulation (GDPR) and the EU AI Act

# Executive Summary

- **Step 1 – Import Libraries:** configurazione dell’ambiente analitico e degli strumenti necessari per il trattamento e la visualizzazione dei dati.

- **Step 2 – Load and Normalize the Dataset:** caricamento e normalizzazione del dataset per renderlo idoneo alle analisi successive.

- **Step 3 – Privacy Identification & Compliance Mapping:** identificazione dei dati personali e classificazione dei rischi in relazione ai requisiti normativi.

- **Step 4 – Privacy Control: Pseudonymization & Data Minimization:** applicazione di controlli di privacy per ridurre l’esposizione dei dati sensibili.

- **Step 5 – Fairness Analysis & Regulatory Impact:** valutazione di possibili bias algoritmici e del loro impatto regolatorio.

- **Step 6 – Right to Erasure:** considerazione dei meccanismi necessari per garantire il diritto alla cancellazione dei dati.

- **Step 7 – EU AI Act Classification & Compliance:** classificazione del sistema come AI ad alto rischio e definizione delle principali implicazioni di governance.

## 1 - Import libraries

We import the standard libraries for data processing and visualization.

These tools enable dataset inspection, fairness metric computation, and governance reporting through plots and tables.

In [8]:
import pandas as pd # Data loading and manipulation
import numpy as np # Data loading and manipulation
import matplotlib.pyplot as plt  # Data loading and manipulation
import hashlib # Hashing for pseudonymization of sensitive data
from fairlearn.metrics import demographic_parity_difference  # Metric for fairness and disparate impact analysis

## 2 - Load and prepare the dataset

The dataset is provided as a cleaned CSV file. We load it using pandas.read_csv() to create a structured DataFrame suitable for analysis.

This step is essential for governance and compliance analysis, as a clear and structured dataset enables proper inspection of all attributes before applying privacy and fairness checks.

In [9]:
DATA_PATH = "../data/clean_credit_applications.csv"
df = pd.read_csv(DATA_PATH)

## 3 — Privacy Identification & Compliance Mapping

The NovaCred system operates within a highly regulated financial sector. For this reason, a governance framework has been established to proactively assess privacy and regulatory risks associated with the data used in the credit scoring model.

This step focuses on the systematic identification and classification of personal data attributes—such as SSNs, emails, and IP addresses—that may expose the organization to regulatory risks under the GDPR.

By mapping each attribute to its risk category and relevant GDPR obligations, the analysis provides clear visibility over sensitive data and supports the implementation of appropriate privacy and fairness controls in later stages of the project.

In [3]:
governance_data = {
    "Attribute": [
        "applicant_info.full_name",
        "applicant_info.ssn",
        "applicant_info.email",
        "applicant_info.ip_address",
        "applicant_info.gender",
        "applicant_info.date_of_birth",
        "applicant_info.zip_code"
    ],
    "Category": [
        "PII (Direct)",
        "PII (Highly Sensitive)",
        "PII (Direct)",
        "PII (Online)",
        "Sensitive Attribute",
        "Sensitive Attribute",
        "Sensitive Proxy"
    ],
    "Risk Type": [
        "Personal identification",
        "Identity exposure",
        "Personal identification",
        "User tracking",
        "Gender discrimination",
        "Age discrimination",
        "Proxy discrimination"
    ],
    "GDPR Obligations": [
        "Art. 6, Art. 5(1)(c) (Minimization)",
        "Art. 6, Art. 5(1)(c), Art. 32 (Requires Hashing)",
        "Art. 6, Art. 5(1)(c) (Minimization)",
        "Art. 6 (Legitimate interest), Art. 5(1)(c)",
        "Art. 9 (Strictly limited, used for bias auditing)",
        "Art. 5(1)(c), Art. 5(1)(e) (Storage limitation)",
        "Art. 5(1)(c) (Data minimization)"
    ]
}

governance_table = pd.DataFrame(governance_data)
display(governance_table)

,Attribute,Category,Risk Type,GDPR Obligations
0,applicant_info.full_name,PII (Direct),Personal identification,"Art. 6, Art. 5(1)(c) (Minimization)"
1,applicant_info.ssn,PII (Highly Sensitive),Identity exposure,"Art. 6, Art. 5(1)(c), Art. 32 (Requires Hashing)"
2,applicant_info.email,PII (Direct),Personal identification,"Art. 6, Art. 5(1)(c) (Minimization)"
3,applicant_info.ip_address,PII (Online),User tracking,"Art. 6 (Legitimate interest), Art. 5(1)(c)"
4,applicant_info.gender,Sensitive Attribute,Gender discrimination,"Art. 9 (Strictly limited, used for bias auditing)"
5,applicant_info.date_of_birth,Sensitive Attribute,Age discrimination,"Art. 5(1)(c), Art. 5(1)(e) (Storage limitation)"
6,applicant_info.zip_code,Sensitive Proxy,Proxy discrimination,Art. 5(1)(c) (Data minimization)


# Analysis of Governance Controls & Risk Identification

The NovaCred governance framework begins with the systematic identification and classification of sensitive attributes contained within the dataset.

Each attribute is evaluated according to its privacy and regulatory risk profile and categorized into three main groups: Personally Identifiable Information (PII), sensitive attributes, and proxy variables. This classification enables the organization to identify potential privacy and discrimination risks associated with automated credit scoring.

Direct identifiers such as full names, SSNs, emails, and IP addresses are classified as PII because they can directly or indirectly enable the identification of an individual. Their processing must comply with GDPR Article 6 (lawfulness of processing) and the data minimization principle under Article 5(1)(c). In the case of highly sensitive identifiers such as SSNs, additional security measures are required under Article 32 (security of processing) to mitigate risks related to identity exposure.

Attributes such as gender and date of birth are considered sensitive because they may influence model outcomes and therefore require strict monitoring to prevent discriminatory effects. In particular, gender falls under the stricter regulatory regime of GDPR Article 9, which governs the processing of special categories of personal data. Meanwhile, date of birth must comply with the data minimization principle (Article 5(1)(c)) and storage limitation requirements under Article 5(1)(e) to ensure that personal data is not retained longer than necessary.

Additionally, variables such as ZIP code are classified as proxy attributes, since they may indirectly reveal protected characteristics such as socioeconomic status or ethnicity. For this reason, their use must be carefully evaluated under the data minimization principle of GDPR Article 5(1)(c) to reduce the risk of indirect discrimination.

By mapping each attribute to its corresponding GDPR obligations, the framework provides a clear legal and governance structure that supports the implementation of privacy-preserving controls and fairness monitoring mechanisms in subsequent stages of the project.


## 4 — Privacy control: Pseudonymization & Data Minimization
In alignment with the regulatory obligations identified in the previous step, privacy risks associated with direct identifiers must be mitigated before conducting further analysis.

To support GDPR-compliant processing, the dataset includes pseudonymized versions of the main direct identifiers, namely full_name, email, ssn, and ip_address. These attributes are transformed into secure hashed representations, generating new fields such as applicant_info.full_name_hash, applicant_info.email_hash, applicant_info.ssn_hash, and applicant_info.ip_address_hash.

This approach preserves analytical consistency by maintaining unique record-level references without exposing raw personally identifiable information during downstream processing.

From a governance perspective, this represents a key privacy control under GDPR Article 32, as pseudonymization reduces the risk of unauthorized identification and limits the impact of potential data exposure.

In addition, the creation of hashed identifiers supports the principle of data minimization under Article 5(1)(c), since analytical workflows can be designed to rely on pseudonymized fields rather than directly processing raw personal data.

Where strictly necessary, the original identifiers may remain temporarily available in the source dataset; however, the working analytical dataset should prioritize hashed columns and exclude unnecessary raw PII whenever possible.

In [19]:
PII_cols = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address"
]

SALT = "DEGO_TXC1"

def salted_sha256(value, salt):
    """
    pseudonymization using salted SHA-256.
    returns man if value is missing.
    """
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if value == "":
        return pd.NA
    return hashlib.sha256((salt + value).encode("utf-8")).hexdigest()

for col in PII_cols:
    df[col + "_hash"] = df[col].apply(lambda x: salted_sha256(x, SALT))

df

,spending_behavior,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,...,financials.savings_balance,decision.loan_approved,decision.interest_rate,decision.approved_amount,applicant_info.age,applicant_info.debt,applicant_info.email_hash,applicant_info.full_name_hash,applicant_info.ssn_hash,applicant_info.ip_address_hash
0,"[{'category': 'Fitness', 'amount': 576}]",Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230.0,102000.0,37.0,...,0.0,False,NaN,NaN,39.0,42840.0,31945e2f062e8351d18141dc9dcf6d27ca65a9127a9672...,5026fdc8343885692958296eb24d73444a12d0fd447dec...,097bd0eb496bb5c9f3ab4f603c70f3babee1ba0588c967...,38207bb99edcd353e72f8bfbeee6ddfa496018449e88a6...
1,"[{'category': 'Education', 'amount': 533}]",Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,Male,1999-08-01,10020.0,41000.0,5.0,...,18200.0,False,NaN,NaN,26.0,14760.0,62dbdc29bbe0e3aa1c38d47c4b8e0574ad2637b2afe36e...,bde5029a73ba0cda33dfe436ce66c6eac995f37bbb6304...,7db610cdd4e41500f939c0ed3428a14295cd53a4a03678...,49590b1aa7450fd133631aa408324aea9860ce94420fe5...
2,"[{'category': 'Healthcare', 'amount': 450}]",Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,Female,1982-08-24,90213.0,65000.0,74.0,...,7090.0,True,3.4,76000.0,43.0,27950.0,1bae736990c1ddfd9b2b29286173166ff970249e9cf372...,69f12c6006bdfbabcca88b4741a32edcd4234e27a12a80...,18cb54d2460f3a1a5676f7288480ef5a20b82528ab01c9...,23955b889f18fc61fd5e0885ea9067554d274f044af473...
3,"[{'category': 'Transportation', 'amount': 329}...",Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,Female,NaN,90217.0,69000.0,9.0,...,10327.0,False,NaN,NaN,NaN,28290.0,b782b8360c90fe97d7dea87eb2c9a09687a661c4040842...,7eadc5af6c04194312428cf912909b9f90af72577740b7...,e557028852c825800d964c65832ad82d3209072aa91a72...,ea707ee9ea23758b206e4092b48209e5c5144d50fc6090...
4,"[{'category': 'Insurance', 'amount': 585}]",Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,Female,NaN,90296.0,39000.0,76.0,...,15011.0,False,NaN,NaN,NaN,2340.0,b0f1b3ae8b62322d2778e827eadd8790d066228800bc4c...,e039526cea40bb9b8e9b368a7afef506be933f95ef3a2f...,d38047bfd248017ab137c868cdb194bd876041a9d29146...,de0477a984d625c0fc3d11df01e833703f07f56f53c79e...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,"[{'category': 'Fitness', 'amount': 175}]",Rachel Miller,rachel.miller69@protonmail.com,953-69-6408,10.194.95.93,Female,1979-11-12,90211.0,87000.0,40.0,...,23949.0,True,6.2,25000.0,46.0,26970.0,1ea7fae6858cc2d2bc02aa2e66a484368ac56900402f38...,a09e2d14663df860081de9875ab833bf3be839c35af6f9...,ea85fb090883442beac52cf0e769a296982ccbfb1d2708...,6fac6a8093482d7174b1e41a997a4c66aa73150602f900...
496,"[{'category': 'Groceries', 'amount': 423}]",Patrick Wilson,patrick.wilson77@hotmail.com,655-58-3025,10.131.43.18,Male,NaN,90233.0,48000.0,4.0,...,21540.0,False,NaN,NaN,NaN,4800.0,b9551fc92c2baf3b1176eef460efa1826882ef40df8fcb...,590cf7b665dc255a5fa6241183e4b8edc06ba37a43d83a...,3cad46c309ba5695746b089188f4c2809c6f619651b0d5...,7f0e6207f726687ce03275cc93c65a843e94a521edc242...
497,"[{'category': 'Education', 'amount': 177}, {'c...",James Rivera,james.rivera25@gmail.com,942-34-6834,172.18.221.237,Male,1989-10-19,90276.0,86000.0,33.0,...,36901.0,False,NaN,NaN,36.0,24940.0,f6f27b8502d063a6dca4e611821c41c7e2692dd270b654...,f66c5c1818d641312e411e68731d79ae3aa5e2fa184d7f...,6d056f098c88bd5ff38d7200d3f249128a597c5953e621...,f0e545ef737326c1b79fa877ec9dbc81248a58d59843a6...
498,"[{'category': 'Groceries', 'amount': 810}]",Deborah Lee,deborah.lee74@mail.com,843-60-2218,10.1.111.83,Female,1983-12-02,90218.0,111000.0,3.0,...,78838.0,True,4.1,62000.0,42.0,21090.0,b8aa5a2df4e1a7295c58b03a489ef4738b72a9331587fd...,7a54a2c712ac899e4f8147faa8d7b6edde2b17a0

# 5 – Fairness Analysis & Regulatory Impact (DIR)

Following the technical analysis of loan approval distributions, we evaluated the Disparate Impact Ratio (DIR) performed by the Data Scientist, to identify potential gender biases. I interpret these results as critical compliance signals:

* Risk Threshold (80% Rule): Monitoring the DIR (the ratio of female to male approval rates) is our primary tool for detecting indirect discrimination. A value below 0.8 acts as a “compliance trigger,” requiring immediate mitigation under Art. 5(2) GDPR (Accountability).

* EU AI Act Obligations: Since NovaCred is classified as a High-Risk AI System, any detected disparity mandates the activation of Human-in-the-Loop protocols (Art. 14 AI Act). This ensures that borderline cases are not decided solely by the algorithm but are subject to human review to guarantee equity.

* Proactive Governance: DIR analysis allows us to move beyond mere technical protection (pseudonymization) toward the substantive protection of fundamental rights. This ensures the model does not automate historical biases, remaining aligned with the principles of Transparency and Fairness.

In [20]:
approval_rates = df.groupby("applicant_info.gender")["decision.loan_approved"].mean()

female_rate = approval_rates.loc["Female"]
male_rate   = approval_rates.loc["Male"]

# Approval gap
gap_pp = (male_rate - female_rate) * 100

# Disparate Impact Ratio (DIR)
dir_ratio = female_rate / male_rate

print("Outcome disparity (Approval rate by gender)")
print(approval_rates)

print("\nFemale approval rate: {:.2f}%".format(female_rate * 100))
print("Male approval rate:   {:.2f}%".format(male_rate * 100))
print("Approval gap (Male - Female): {:.2f} percentage points".format(gap_pp))

print("\nDisparate Impact Ratio (DIR): {:.2f}".format(dir_ratio))

# Check 80% rule
if dir_ratio < 0.8:
    print("Warning: DIR below 0.8 → Potential indirect discrimination (80% rule triggered)")
else:
    print("DIR above 0.8 → No significant disparate impact detected")

Outcome disparity (Approval rate by gender)
applicant_info.gender
Female    0.505976
Male      0.662651
Name: decision.loan_approved, dtype: float64

Female approval rate: 50.60%
Male approval rate:   66.27%
Approval gap (Male - Female): 15.67 percentage points

Disparate Impact Ratio (DIR): 0.76


##  6 - Right to Erasure ( GDPR Art. 17)

The GDPR grants data subjects the right to have their personal data erased under specific conditions. For instance, if the model shows signs of potential discrimination, the processing could be deemed unlawful. In such cases, users have the right to demand the permanent deletion of their data to prevent further unfair treatment.

While some financial systems use advanced techniques like anonymization (e.g., crypto-shredding), we are demonstrating a physical deletion approach. This ensures that the requester’s record is entirely removed from the active analytical dataset, strictly adhering to the “Right to be Forgotten.”

This procedure directly supports Data Minimization (Art. 5c) and Storage Limitation (Art. 5e) by ensuring that data is no longer processed once the legal basis is compromised by discriminatory outcomes or when the user’s consent is withdrawn.

In [18]:
if "applicant_info.email_hash" not in df.columns:
    df["applicant_info.email_hash"] = df["applicant_info.email"].astype(str).apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()
    )
target_to_delete = df["applicant_info.email_hash"].iloc[0]

df_privacy = df[df["applicant_info.email_hash"] != target_to_delete].copy()

print("Erasure completed successfully.")
print("Total records remaining in dataset:", len(df_privacy))

Erasure completed successfully.
Total records remaining in dataset: 499


7 – EU AI Act Classification & Regulatory Compliance

7.1 – EU AI Act Classification: High-Risk

Under Annex III (5b) of the EU AI Act, AI systems used for creditworthiness assessment of natural persons are explicitly classified as High-Risk AI Systems. This classification triggers strict mandatory obligations because automated credit decisions significantly impact individuals’ fundamental rights and access to essential services.

⸻

7.2 – Governance Gap Analysis: Lawful Basis & Storage Policy

To ensure full compliance beyond technical protection, we identified two critical governance requirements for the NovaCred data pipeline:

1. Lawful Basis (Art. 6 GDPR)

Our audit revealed the absence of a “Consent Flag” in the raw dataset. To ensure legality, NovaCred operates under:

* Contractual Necessity (Art. 6.1.b): Necessary for the credit risk assessment requested by the applicant. NovaCred must demonstrate that the service cannot exist without these specific data points.

* Explicit Consent (Art. 6.1.a): Required for secondary behavioral analytics. NovaCred cannot use data for accessory purposes without a specific, voluntary opt-in.

* Proposed Control: We recommend adding a technical “Validation Gateway” at the start of the data process. This system will automatically reject any record that does not have a consent_given = True flag and a valid timestamp. For existing data without this flag, our policy is to delete any information that is not strictly necessary for the contract, ensuring NovaCred never processes “orphan” data.


2. Storage Limitation & Retention (Art. 5.1.e GDPR)

Data cannot be stored indefinitely. We propose a tiered retention policy to mitigate long-term liability:

* Active Phase: Full raw data is kept only during the loan application window.

* Archival Phase: Upon a decision (approval/rejection), direct identifiers are purged.

* Statistical Phase: Only the pseudonymized hashes (generated in Step 4) are retained for 5 years to support regulatory audits and bias monitoring. After this period, even the hashes are permanently deleted (Crypto-shredding).

⸻

7.3 – Mandatory Obligations for High-Risk Systems

To comply with the EU AI Act and GDPR, our governance framework must address the following requirements:

Human Oversight (Human-in-the-Loop)

* Mandatory human review for borderline cases and escalation channels for applicants to contest automated denials.

* Art. 22 GDPR (Automated individual decision-making) and Art. 14 EU AI Act (Human oversight).